# Stage 03a — One-animal WildFusion similarity matrix

Kaggle GPU notebook. This is the `new-clustring-analysis.ipynb` extraction/calibration stage adapted to the frozen ApexID CVPR benchmark.

- Choose exactly one `DATASET_NAME` and one `GLOBAL_DESCRIPTOR`.
- WildFusion always includes ALIKED + LightGlue and DISK + LightGlue.
- Calibration uses a fixed 512-image cohort, deterministically stratified across Stage 00 datasets and persisted with a SHA-256 hash.
- The selected animal's full train/train similarity matrix is saved as `sim_matrix_{GLOBAL_DESCRIPTOR}_{DATASET_NAME}_full.npy`.
- This notebook stops after writing and validating that artifact. Notebook 03b performs Optuna.

In [ ]:
# Kaggle dependencies (internet must be enabled for first model/package download).
%pip install -q git+https://github.com/WildlifeDatasets/wildlife-datasets@develop
%pip install -q git+https://github.com/WildlifeDatasets/wildlife-tools
%pip install -q "transformers==4.45.2" timm albumentations networkx huggingface_hub

import gc
import hashlib
import json
import os
import subprocess
import sys
from pathlib import Path

import numpy as np
import pandas as pd
import torch
import torchvision.transforms as T
from PIL import Image

ON_KAGGLE = Path("/kaggle/working").is_dir()
if ON_KAGGLE and not Path("/kaggle/working/ApexID/apexid").is_dir():
    subprocess.check_call(["git", "clone", "-q", "https://github.com/MohamedBassat/ApexID.git", "/kaggle/working/ApexID"])
REPO_ROOT = Path("/kaggle/working/ApexID" if ON_KAGGLE else Path.cwd()).resolve()
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))

from wildlife_datasets.datasets import WildlifeDataset
from wildlife_tools.features import DeepFeatures
from wildlife_tools.features.local import AlikedExtractor, DiskExtractor
from wildlife_tools.similarity import CosineSimilarity
from wildlife_tools.similarity.calibration import IsotonicCalibration
from wildlife_tools.similarity.pairwise.lightglue import MatchLightGlue
from wildlife_tools.similarity.wildfusion import SimilarityPipeline, WildFusion

print("torch:", torch.__version__, "cuda:", torch.cuda.is_available())
if not torch.cuda.is_available():
    raise RuntimeError("03a requires a Kaggle GPU: global extraction and LightGlue run here.")

In [ ]:
# ============================================================
# USER KNOBS — one animal × one global descriptor per session
# ============================================================
DATASET_NAME = "jaguar_antonio"
GLOBAL_DESCRIPTOR = "ApexID_BHCA_A_v2"

ALLOWED_DATASETS = (
    "bristol_gorillas",
    "spac_wild",
    "elephants",
    "gcn_id",
    "jaguar_antonio",
)
ALLOWED_GLOBAL_DESCRIPTORS = (
    "ApexID_BHCA_A_v2",
    "ApexID_BHCA_SCA_v2",
    "ApexID_BHCA_SCSF2_v2",
    "MiewIDv3",
    "MegaDescriptor",
)
if DATASET_NAME not in ALLOWED_DATASETS:
    raise ValueError(f"DATASET_NAME must be one of {ALLOWED_DATASETS}; got {DATASET_NAME!r}")
if GLOBAL_DESCRIPTOR not in ALLOWED_GLOBAL_DESCRIPTORS:
    raise ValueError(f"GLOBAL_DESCRIPTOR must be one of {ALLOWED_GLOBAL_DESCRIPTORS}; got {GLOBAL_DESCRIPTOR!r}")

B = 256
GLOBAL_BATCH_SIZE = 16
LOCAL_BATCH_SIZE = 16
CALIBRATION_SIZE = 512
CALIBRATION_SEED = 42
# Every model/animal run uses the same deterministic, dataset-stratified cohort.
CALIBRATION_SAMPLE_NAME = "wildfusion_calibration_sample_512.csv"
DEVICE = torch.device("cuda")

STAGE00_ROOT_CANDIDATES = (
    Path(os.environ.get("APEXID_PREPARED_INPUT", "")),
    Path("/kaggle/input/datasets/seifelkerdany/apexid-cvpr-prepared-bundle/apexid_cvpr_prepared"),
    Path("/kaggle/input/apexid-cvpr-prepared-bundle/apexid_cvpr_prepared"),
    Path("/kaggle/working/apexid_cvpr_prepared"),
)
MODEL_PATHS = {
    "ApexID_BHCA_A_v2": Path("/kaggle/input/models/mohamedbassat/apexid-convnext-small-bhca-a-v2-rgb/pytorch/default/1/apexid_convnext_small_bhca_a_v2_rgb/best_backbone.pt"),
    "ApexID_BHCA_SCA_v2": Path("/kaggle/input/models/mohamedbassat/apexid-convnext-small-bhca-sca-v2-rgb/pytorch/default/1/apexid_convnext_small_bhca_sca_v2_rgb/best_backbone.pt"),
    "ApexID_BHCA_SCSF2_v2": Path("/kaggle/input/models/mohamedbassat/apexid-convnext-small-bhca-scsf2-v2-rgb/pytorch/default/1/apexid_convnext_small_bhca_scsf2_v2_rgb/best_backbone.pt"),
}
SIDECAR_CONFIG_NAMES = ("config_resolved.yaml", "config_resolved.yml", "config.yaml", "config.yml")
OUTPUT_ROOT = Path(os.environ.get(
    "APEXID_STAGE03A_OUTPUT",
    f"/kaggle/working/apexid_cvpr_stage03/03a/{GLOBAL_DESCRIPTOR}/{DATASET_NAME}" if ON_KAGGLE
    else str(REPO_ROOT / "outputs" / "stage03" / "03a" / GLOBAL_DESCRIPTOR / DATASET_NAME),
)).resolve()
OUTPUT_ROOT.mkdir(parents=True, exist_ok=True)


def first_root(candidates, marker):
    for candidate in candidates:
        if str(candidate) and (candidate / marker).is_file():
            return candidate.resolve()
    raise FileNotFoundError(f"Could not find {marker}; checked: {[str(x) for x in candidates]}")

STAGE00_ROOT = first_root(STAGE00_ROOT_CANDIDATES, "preparation_manifest.json")
print("DATASET_NAME:", DATASET_NAME)
print("GLOBAL_DESCRIPTOR:", GLOBAL_DESCRIPTOR)
print("STAGE00_ROOT:", STAGE00_ROOT)
print("OUTPUT_ROOT:", OUTPUT_ROOT)

In [ ]:
# Load frozen manifests and resolve their original paths to this Kaggle session.
PREPARATION_MANIFEST = json.loads((STAGE00_ROOT / "preparation_manifest.json").read_text(encoding="utf-8"))
if PREPARATION_MANIFEST.get("status") != "passed":
    raise RuntimeError("Stage 00 preparation_manifest.json is not marked passed")

MANIFEST_PATHS = {
    "apexid_calibration": STAGE00_ROOT / "manifests" / "apexid_calibration_manifest.csv",
    **{name: STAGE00_ROOT / "manifests" / f"{name}_canonical_manifest.csv" for name in ALLOWED_DATASETS},
}
LIVE_IMAGE_ROOTS = (
    Path("/kaggle/input/datasets/seifelkerdany/apexid-dataset/apexid-dataset"),
    Path("/kaggle/input/datasets/seifelkerdany/apexid-dataset/apexid-dataset/images"),
    Path("/kaggle/input/datasets/jamalhussien/bristol-gorillas-sam3-black-background"),
    Path("/kaggle/input/datasets/jamalhussien/bristol-gorillas-sam3-black-background/images"),
    Path("/kaggle/input/datasets/jamalhussien/apexid-wildlife-reid-test-data"),
    Path("/kaggle/input/datasets/jamalhussien/apexid-wildlife-reid-test-data/images"),
    Path("/kaggle/input/datasets/seifelkerdany/apexid-jaguar-materialized"),
    Path("/kaggle/input/apexid-jaguar-materialized"),
    Path("/kaggle/working/apexid_jaguar_materialized"),
)


def resolve_image_path(frozen_path: str) -> Path:
    raw = str(frozen_path).replace("\\", "/")
    direct = Path(raw)
    if direct.is_file():
        return direct.resolve()
    tail = raw.split("/images/", 1)[1] if "/images/" in raw else "/".join(Path(raw).parts[-2:])
    identity, filename = Path(raw).parent.name, Path(raw).name
    for root in LIVE_IMAGE_ROOTS:
        for candidate in (root / tail, root / "images" / tail, root / identity / filename, root / "images" / identity / filename, root / filename):
            if candidate.is_file():
                return candidate.resolve()
    raise FileNotFoundError(
        f"Cannot resolve frozen image {frozen_path!r}. Attach the same ApexID/Bristol/wildlife/Jaguar image datasets used by notebooks 00/01. "
        "For jaguar_antonio, attach the materialized FiftyOne body-mask dataset (or run notebook 01's Jaguar materialization first)."
    )

frames = {}
for dataset_name, path in MANIFEST_PATHS.items():
    if not path.is_file():
        raise FileNotFoundError(path)
    frame = pd.read_csv(path, dtype=str).fillna("")
    required = {"image_id", "path", "identity", "dataset"}
    if not required <= set(frame.columns):
        raise ValueError(f"{dataset_name}: missing columns {sorted(required - set(frame.columns))}")
    if frame["image_id"].duplicated().any():
        raise ValueError(f"{dataset_name}: duplicate image_id")
    frames[dataset_name] = frame.reset_index(drop=True)

# Resolve only one sample per dataset first, so missing Kaggle inputs fail immediately.
for name, frame in frames.items():
    print(name, "sample ->", resolve_image_path(frame.iloc[0]["path"]))

# WildlifeDataset needs paths relative to root or absolute. Identity labels are namespaced
# during mixed-species calibration so e.g. identity '1' in two datasets is not a positive pair.
def wildlife_frame(frame: pd.DataFrame, *, namespace_identity: bool) -> pd.DataFrame:
    out = frame[["image_id", "path", "identity", "dataset"]].copy()
    out["frozen_path"] = out["path"].astype(str)
    out["path"] = [str(resolve_image_path(path)) for path in out["path"]]
    if namespace_identity:
        out["identity"] = out["dataset"].astype(str) + "::" + out["identity"].astype(str)
    return out

full_calibration_catalogue = pd.concat(
    [wildlife_frame(frame, namespace_identity=True) for frame in frames.values()],
    ignore_index=True,
)
selected_catalogue = wildlife_frame(frames[DATASET_NAME], namespace_identity=False)

# Reuse an attached cohort when available; otherwise create the same stratified
# sample deterministically from the frozen manifests. Sampling is balanced as
# evenly as possible across datasets, then filled from the remaining pool.
def calibration_sample_path_candidates():
    candidates = [OUTPUT_ROOT / CALIBRATION_SAMPLE_NAME]
    if Path("/kaggle/input").is_dir():
        candidates.extend(Path("/kaggle/input").glob(f"**/{CALIBRATION_SAMPLE_NAME}"))
    return candidates


def deterministic_stratified_sample(catalogue, size, seed):
    if size > len(catalogue):
        raise ValueError(f"CALIBRATION_SIZE={size} exceeds catalogue size {len(catalogue)}")
    rng = np.random.default_rng(seed)
    dataset_names = sorted(catalogue["dataset"].astype(str).unique())
    base, remainder = divmod(size, len(dataset_names))
    selected_indices = []
    for rank, dataset_name in enumerate(dataset_names):
        pool = catalogue.index[catalogue["dataset"].eq(dataset_name)].to_numpy()
        take = min(len(pool), base + int(rank < remainder))
        selected_indices.extend(rng.choice(pool, size=take, replace=False).tolist())
    if len(selected_indices) < size:
        remaining = np.setdiff1d(catalogue.index.to_numpy(), np.asarray(selected_indices), assume_unique=False)
        selected_indices.extend(rng.choice(remaining, size=size - len(selected_indices), replace=False).tolist())
    sampled = catalogue.loc[selected_indices].copy()
    # Stable order makes the calibration pair stream identical in every run.
    return sampled.sort_values(["dataset", "image_id"], kind="stable").reset_index(drop=True)

existing_sample = next((path for path in calibration_sample_path_candidates() if path.is_file()), None)
if existing_sample is not None:
    frozen_sample = pd.read_csv(existing_sample, dtype=str).fillna("")
    required_sample = {"dataset", "image_id"}
    if not required_sample <= set(frozen_sample.columns):
        raise ValueError(f"Calibration sample {existing_sample} lacks {required_sample}")
    requested_keys = list(zip(frozen_sample["dataset"], frozen_sample["image_id"]))
    catalogue_by_key = full_calibration_catalogue.set_index(["dataset", "image_id"], drop=False)
    missing_keys = [key for key in requested_keys if key not in catalogue_by_key.index]
    if missing_keys:
        raise RuntimeError(f"Persisted calibration sample has IDs absent from Stage 00: {missing_keys[:5]}")
    calibration_catalogue = catalogue_by_key.loc[requested_keys].reset_index(drop=True)
    print("Reused persisted calibration cohort:", existing_sample)
else:
    calibration_catalogue = deterministic_stratified_sample(
        full_calibration_catalogue, CALIBRATION_SIZE, CALIBRATION_SEED
    )

if len(calibration_catalogue) != CALIBRATION_SIZE:
    raise RuntimeError("Calibration cohort size changed")
calibration_sample_columns = ["dataset", "image_id", "identity"]
calibration_sample_path = OUTPUT_ROOT / CALIBRATION_SAMPLE_NAME
calibration_catalogue[calibration_sample_columns].to_csv(calibration_sample_path, index=False)
# Hash logical membership, not machine-specific absolute image paths or CSV newlines.
calibration_membership = calibration_catalogue[["dataset", "image_id"]].astype(str).to_dict(orient="records")
calibration_sample_sha256 = hashlib.sha256(
    json.dumps(calibration_membership, sort_keys=True, separators=(",", ":")).encode("utf-8")
).hexdigest()
print(calibration_catalogue.groupby("dataset").size())
print("Calibration cohort SHA256:", calibration_sample_sha256)

class FrozenStage00Dataset(WildlifeDataset):
    summary = {"animals": ["mixed wildlife"]}

calibration_dataset = FrozenStage00Dataset(root="/", df=calibration_catalogue, check_files=True, load_label=True)
selected_dataset = FrozenStage00Dataset(root="/", df=selected_catalogue, check_files=True, load_label=True)
print("Calibration images:", len(calibration_dataset))
print("Selected images:", len(selected_dataset), "identities:", selected_catalogue["identity"].nunique())

In [ ]:
# Build the selected global descriptor. ALIKED/DISK remain identical for every ablation.
import torch.nn.functional as F


class EmbeddingModel(torch.nn.Module):
    def __init__(self, core):
        super().__init__()
        self.core = core

    def forward(self, x):
        out = self.core(x)
        if isinstance(out, dict):
            if "embeddings" in out:
                out = out["embeddings"]
            elif "last_hidden_state" in out:
                out = out["last_hidden_state"].mean(dim=1)
            else:
                out = next(iter(out.values()))
        if out.ndim > 2:
            out = out.reshape(out.shape[0], -1)
        return F.normalize(out.float(), dim=1)


class AlbumentationsFromPIL:
    def __init__(self, transform):
        self.transform = transform

    def __call__(self, image: Image.Image):
        return self.transform(image=np.asarray(image.convert("RGB")))["image"]


RGB_BHCA_EVAL_CONFIG = {
    "seed": 42,
    "data": {
        "image_size": [440, 440], "wavelet": False, "grayscale": False,
        "transforms": {"mean": [0.485, 0.456, 0.406], "std": [0.229, 0.224, 0.225]},
    },
    "model": {
        "name": "apexid", "embedding_dim": 2048, "input_channels": 3,
        "backbone": {"name": "convnext_small_pt", "kwargs": {"pretrained": True}},
    },
}


def find_sidecar(weights: Path, names):
    for folder in (weights.parent, weights.parent.parent, weights.parent.parent.parent):
        if not folder.is_dir():
            continue
        for filename in names:
            candidate = folder / filename
            if candidate.is_file():
                return candidate.resolve()
    return None


def sha256_file(path: Path):
    digest = hashlib.sha256()
    with path.open("rb") as handle:
        for chunk in iter(lambda: handle.read(8 * 1024 * 1024), b""):
            digest.update(chunk)
    return digest.hexdigest()


def build_global_descriptor(name: str):
    if name.startswith("ApexID_BHCA_"):
        from apexid.data import build_eval_transforms
        from apexid.evaluation.benchmark_evaluator import ApexIDModelAdapter
        from apexid.utils.config import load_config

        weights = MODEL_PATHS[name]
        if not weights.is_file():
            raise FileNotFoundError(f"Attach the Kaggle model for {name}: {weights}")
        config_path = find_sidecar(weights, SIDECAR_CONFIG_NAMES)
        cfg = (
            load_config(config_path=config_path)
            if config_path is not None
            else load_config(config=RGB_BHCA_EVAL_CONFIG)
        )
        adapter = ApexIDModelAdapter(cfg, DEVICE, amp=True)
        adapter.load_weights(weights)
        transform = AlbumentationsFromPIL(build_eval_transforms(config=cfg))
        model = EmbeddingModel(adapter.model).to(DEVICE).eval()
        provenance = {
            "weights": str(weights),
            "weights_sha256": adapter.checkpoint_metadata["weights_sha256"],
            "config_path": None if config_path is None else str(config_path),
            "config_sha256": (
                sha256_file(config_path)
                if config_path is not None
                else hashlib.sha256(json.dumps(RGB_BHCA_EVAL_CONFIG, sort_keys=True).encode("utf-8")).hexdigest()
            ),
            "config_source": "sidecar" if config_path is not None else "rgb_bhca_eval_fallback_imagenet",
        }
        return model, transform, provenance

    if name == "MiewIDv3":
        from transformers import AutoModel
        core = AutoModel.from_pretrained("conservationxlabs/miewid-msv3", trust_remote_code=True)
        model = EmbeddingModel(core).to(DEVICE).eval()
        transform = T.Compose([
            T.Resize([440, 440]), T.ToTensor(),
            T.Normalize(mean=(0.485, 0.456, 0.406), std=(0.229, 0.224, 0.225)),
        ])
        return model, transform, {"hf_tag": "conservationxlabs/miewid-msv3"}

    if name == "MegaDescriptor":
        import timm
        tag = "hf-hub:BVRA/MegaDescriptor-L-384"
        core = timm.create_model(tag, num_classes=0, pretrained=True)
        cfg = timm.data.resolve_model_data_config(core)
        size = int(cfg.get("input_size", (3, 384, 384))[-1])
        transform = T.Compose([
            T.Resize([size, size]), T.ToTensor(),
            T.Normalize(mean=cfg.get("mean", (0.485, 0.456, 0.406)), std=cfg.get("std", (0.229, 0.224, 0.225))),
        ])
        return EmbeddingModel(core).to(DEVICE).eval(), transform, {"timm_tag": tag, "image_size": size}

    raise AssertionError(name)


global_model, global_transform, global_provenance = build_global_descriptor(GLOBAL_DESCRIPTOR)
local_transform = T.Compose([T.Resize([512, 512]), T.ToTensor()])

matcher_global = SimilarityPipeline(
    matcher=CosineSimilarity(),
    extractor=DeepFeatures(model=global_model, device=str(DEVICE), batch_size=GLOBAL_BATCH_SIZE),
    transform=global_transform,
    calibration=IsotonicCalibration(),
)
matcher_aliked = SimilarityPipeline(
    matcher=MatchLightGlue(features="aliked", device=str(DEVICE), batch_size=LOCAL_BATCH_SIZE),
    extractor=AlikedExtractor(), transform=local_transform, calibration=IsotonicCalibration(),
)
matcher_disk = SimilarityPipeline(
    matcher=MatchLightGlue(features="disk", device=str(DEVICE), batch_size=LOCAL_BATCH_SIZE, init_threshold=0.9),
    extractor=DiskExtractor(), transform=local_transform, calibration=IsotonicCalibration(),
)
wildfusion = WildFusion(
    calibrated_pipelines=[matcher_aliked, matcher_global, matcher_disk],
    priority_pipeline=matcher_global,
)
print("Global descriptor ready:", GLOBAL_DESCRIPTOR, global_provenance)

In [ ]:
# Deterministic 512-image, cross-dataset isotonic calibration cohort.
print("Calibrating WildFusion on", len(calibration_dataset), "persisted images ...")
print("Calibration cohort SHA256:", calibration_sample_sha256)
wildfusion.fit_calibration(calibration_dataset, calibration_dataset)
print("Calibration complete")

# Compute one animal only. This is the single expensive artifact consumed by 03b/03c.
print(f"Computing {GLOBAL_DESCRIPTOR}/{DATASET_NAME}: {len(selected_dataset)} x {len(selected_dataset)}")
similarity = wildfusion(selected_dataset, selected_dataset, B=B)
similarity = np.asarray(similarity, dtype=np.float32)
similarity[~np.isfinite(similarity)] = 0.0
similarity = np.clip(similarity, 0.0, 1.0)
if similarity.shape != (len(selected_dataset), len(selected_dataset)):
    raise RuntimeError(f"Unexpected matrix shape {similarity.shape}")
if not np.allclose(similarity, similarity.T, atol=2e-4, rtol=0):
    raise RuntimeError("Full train/train WildFusion matrix is not symmetric")
np.fill_diagonal(similarity, 1.0)

image_ids = selected_catalogue["image_id"].astype(str).to_numpy(dtype="U")
identities = selected_catalogue["identity"].astype(str).to_numpy(dtype="U")
paths = selected_catalogue["frozen_path"].astype(str).to_numpy(dtype="U")
bundle = {
    DATASET_NAME: {
        "sim": similarity,
        "indices": np.arange(len(selected_catalogue), dtype=np.int64),
        "image_ids": image_ids,
        "identities": identities,
        "paths": paths,
        "dataset": DATASET_NAME,
        "global_descriptor": GLOBAL_DESCRIPTOR,
        "global_provenance": global_provenance,
        "wildfusion": {
            "local_branches": ["ALIKED+LightGlue", "DISK+LightGlue"],
            "B": B,
            "calibration_images": len(calibration_dataset),
            "calibration_seed": CALIBRATION_SEED,
            "calibration_sample_sha256": calibration_sample_sha256,
        },
    }
}
output_path = OUTPUT_ROOT / f"sim_matrix_{GLOBAL_DESCRIPTOR}_{DATASET_NAME}_full.npy"
tmp_path = output_path.with_suffix(".tmp.npy")
np.save(tmp_path, bundle, allow_pickle=True)
tmp_path.replace(output_path)
print("Saved:", output_path, f"({output_path.stat().st_size / 2**30:.2f} GiB)")

In [ ]:
# Reload validation and hand-off manifest. No Optuna or clustering runs in 03a.
loaded = np.load(output_path, allow_pickle=True).item()[DATASET_NAME]
assert loaded["sim"].shape == similarity.shape
assert np.array_equal(loaded["image_ids"].astype(str), image_ids.astype(str))
assert np.array_equal(loaded["identities"].astype(str), identities.astype(str))

manifest = {
    "stage": "03a_wildfusion_full_matrix",
    "status": "passed",
    "dataset": DATASET_NAME,
    "global_descriptor": GLOBAL_DESCRIPTOR,
    "matrix_path": output_path.name,
    "matrix_shape": list(similarity.shape),
    "matrix_dtype": str(similarity.dtype),
    "matrix_sha256": hashlib.sha256(output_path.read_bytes()).hexdigest(),
    "num_images": int(len(image_ids)),
    "num_identities": int(pd.Series(identities).nunique()),
    "calibration_images": int(len(calibration_dataset)),
    "calibration_scope": "deterministic_dataset_stratified_stage00_subset",
    "calibration_seed": CALIBRATION_SEED,
    "calibration_sample_file": CALIBRATION_SAMPLE_NAME,
    "calibration_sample_sha256": calibration_sample_sha256,
    "calibration_dataset_counts": {
        str(name): int(count) for name, count in calibration_catalogue.groupby("dataset").size().items()
    },
    "calibration_selected_dataset_overlap_images": int(calibration_catalogue["dataset"].eq(DATASET_NAME).sum()),
    "calibration_protocol_note": (
        "Transductive reference-compatible calibration: the fixed generic Stage 00 cohort may include labelled "
        "images from the selected dataset. Do not describe final ARI as calibration-leakage-free."
    ),
    "local_branches": ["ALIKED+LightGlue", "DISK+LightGlue"],
    "B": B,
    "global_provenance": global_provenance,
}
manifest_path = OUTPUT_ROOT / "stage03a_manifest.json"
manifest_path.write_text(json.dumps(manifest, indent=2, sort_keys=True), encoding="utf-8")
print(json.dumps(manifest, indent=2))
print(f"\nNEXT: Save {OUTPUT_ROOT} as a Kaggle dataset, then attach it to notebook 03b with the same DATASET_NAME and GLOBAL_DESCRIPTOR.")